In [2]:
import pymysql
from configparser import ConfigParser

config = ConfigParser()
config.read('../Chapter1/config.ini') # 指定設定檔的檔案路徑

connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
)

print(connection.open)

True


建立資料庫

In [10]:
database = "chapter2"
with connection.cursor() as cursor:
    sql = f"""
        create database if not exists {database}
    """
    # 執行建立的 SQL 語句
    cursor.execute(sql)
    # 執行查看資料庫
    cursor.execute("show databases;")
    dbs = cursor.fetchall()
print(dbs)

[{'Database': 'chapter2'}, {'Database': 'information_schema'}, {'Database': 'my_database'}, {'Database': 'my_titanic'}, {'Database': 'my_train_titanic'}, {'Database': 'mysql'}, {'Database': 'performance_schema'}, {'Database': 'sakila'}, {'Database': 'social_media_app'}, {'Database': 'sys'}, {'Database': 'transaction_test'}, {'Database': 'world'}]


建立資料表

In [4]:
connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database=database,
)

""" 建立 user 資料表
    id 主鍵
    name 字串 不能為空
    age 整數 
    username 字串 不能為空 必須唯一
    password 字串 不能為空
"""
with connection.cursor() as cursor:
    sql = """
        CREATE TABLE IF NOT EXISTS user (
            id INT AUTO_INCREMENT PRIMARY KEY,
            name VARCHAR(250) NOT NULL,
            age INT,
            username VARCHAR(250) NOT NULL UNIQUE,
            password VARCHAR(20) NOT NULL
        );
    """
    cursor.execute(sql)
    cursor.execute("SHOW TABLES")
    tables = cursor.fetchall()

print(tables)


[{'Tables_in_chapter2': 'user'}]


寫入資料

In [ ]:
from pprint import pprint
with connection.cursor() as cursor:
    sql = """
        INSERT INTO user(name, age, username, password)
        VALUES ("Alex", 20, "Alex1", '123')
    """
    # 執行寫入的 SQL 語句
    cursor.execute(sql)
    
    # 執行查詢資料表
    cursor.execute("SELECT * FROM user;")  #在執行 connection.commit() 前，資料只存在記憶體中
    # 取得查詢的所有資料
    r = cursor.fetchall
print(r)

<bound method Cursor.fetchall of <pymysql.cursors.DictCursor object at 0x000001A7358FBDA0>>


In [ ]:
connection.commit()  #執行後才確認進入資料庫

使用另一個連線查詢資料庫

In [21]:
connection2 = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database=database,
)
with connection2.cursor() as cursor:
    cursor.execute("SELECT * FROM user;")
    result = cursor.fetchall()

pprint(result)
connection2.close()

[{'age': 35, 'id': 3, 'name': 'Alex', 'password': '123', 'username': 'Alex1'},
 {'age': 35, 'id': 4, 'name': 'Alice', 'password': '123', 'username': 'Alice1'}]


提交資料庫變更 `conncection.commit()`

In [12]:
from pprint import pprint
with connection.cursor() as cursor:
    sql = """
        INSERT INTO user(name, age, username, password)
        VALUES ("Alice", 20, "Alice1", '123')
    """
    cursor.execute(sql)
    
    # 提交資料庫的變更
    row = cursor.rowcount  #異動的行數
    print(f"資料庫異動: {row} 行")
    if row == 1:
        connection.commit()    

IntegrityError: (1062, "Duplicate entry 'Alice1' for key 'user.username'")

In [13]:
from pprint import pprint
from pymysql.err import IntegrityError

with connection.cursor() as cursor:
    sql = """
        INSERT INTO user(name, age, username, password)
        VALUES ("Alice", 20, "Alice1", '123')
    """
    try:
        cursor.execute(sql)
        
        # 提交資料庫的變更
        row = cursor.rowcount  #異動的行數
        print(f"資料庫異動: {row} 行")
        if row == 1:
            connection.commit()
    except IntegrityError:
        print("該 user 已建立")  
     

該 user 已建立


更新資料

In [ ]:
with connection.cursor() as cursor:
    sql = """
        UPDATE user SET age = 35
        WHERE username = 'Alex1';  
    """
    cursor.execute(sql)
    
    # 可以搭配 rowcount 屬性來檢查受影響的行數
    row = cursor.rowcount
    if row == 1:
        # 提交資料庫的變更
        connection.commit()
    else:
        print("未改變資料")
    
    cursor.execute("SELECT * FROM user;")
    result = cursor.fetchall()

pprint(result)

[{'age': 35, 'id': 3, 'name': 'Alex', 'password': '123', 'username': 'Alex1'},
 {'age': 20, 'id': 4, 'name': 'Alice', 'password': '123', 'username': 'Alice1'}]


In [20]:
with connection.cursor() as cursor:
    sql = """
        UPDATE user SET age = 35
        WHERE id = 3;  
    """
    cursor.execute(sql)
    
    # 可以搭配 rowcount 屬性來檢查受影響的行數
    row = cursor.rowcount
    if row == 1:
        # 提交資料庫的變更
        connection.commit()
    else:
        connection.rollback()   #取消該連線做的任何修改
        print("未改變資料")
    
    cursor.execute("SELECT * FROM user;")
    result = cursor.fetchall()

pprint(result)

未改變資料
[{'age': 35, 'id': 3, 'name': 'Alex', 'password': '123', 'username': 'Alex1'},
 {'age': 35, 'id': 4, 'name': 'Alice', 'password': '123', 'username': 'Alice1'}]


刪除資料表

In [ ]:
import pymysql
from configparser import ConfigParser

config = ConfigParser()
config.read('../Chapter1/config.ini') # 指定設定檔的檔案路徑

connection = pymysql.connect(
    host=config.get('DB', 'host'),
    user=config.get('DB', 'user'),
    password=config.get('DB', 'password'),
    port=config.getint('DB', 'port'),
    cursorclass=pymysql.cursors.DictCursor,
    database="Chapter2"
)



with connection.cursor() as cursor:
    sql = """
        DROP TABLE IF EXISTS user; 
    """
    cursor.execute(sql)
    cursor.execute("SHOW TABLES;")
    result = cursor.fetchall()

print(result)

In [1]:
connection.close()

NameError: name 'connection' is not defined